# Stage 1 — Unsupervised Anomaly Baseline (full normal-sea baseline, prawn-only crosscheck)

Changes from the previous version:
- Baseline is now built from **all** images in the normal-sea baseline split (no subsampling / no convergence check needed - more data gives a more stable reference).
- Negative controls use the **full** held-out normal-sea split, not a capped sample.
- Crosscheck uses **prawn positives only** - tuna/cetacean bonus checks removed for this run.
- Feature extraction is parallelized (`ProcessPoolExecutor`) since we're now processing the full dataset rather than a subsample.

In [24]:
# --- Cell 1: Imports and config ---
import cv2
import numpy as np
import random
import json
from pathlib import Path
from skimage.segmentation import slic
from skimage.feature import graycomatrix, graycoprops
from scipy import ndimage
from concurrent.futures import ProcessPoolExecutor

# --- Dataset paths ---
BASELINE_DIR  = Path("data/normal_sea/baseline_split")   # ALL images here build "what is normal water"
NEGATIVE_DIR  = Path("data/normal_sea/heldout_split")     # ALL images here are negative controls
PRAWN_DIR     = Path("data/prawns")             # deduplicated 42 unique prawn images - crosscheck only

IMG_SIZE = (512, 512)
N_SEGMENTS = 150
GLCM_LEVELS = 32
MIN_COMPONENT_SIZE = 3         # minimum flagged superpixels to count as a real detection
Z_THRESH = 2.5

random.seed(42)

In [13]:
# --- Cell 2: Pre-processing functions ---
# Dataset used: none - applied identically to every set later.

def load_and_resize(path, size=IMG_SIZE):
    img = cv2.imread(str(path))
    img = cv2.resize(img, size)
    return img

def suppress_glint(img_bgr, thresh=220, dilate_iter=2):
    """Masks out bright sun-glint pixels on the L channel."""
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    L = lab[:, :, 0]
    glint_mask = (L > thresh).astype(np.uint8)
    glint_mask = cv2.dilate(glint_mask, np.ones((5, 5), np.uint8), iterations=dilate_iter)
    return glint_mask  # 1 = glint, exclude from analysis

def to_lab(img_bgr):
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)

def preprocess_image(path):
    """Full chain: load -> resize -> LAB -> glint mask."""
    img = load_and_resize(path)
    lab = to_lab(img)
    glint_mask = suppress_glint(img)
    return img, lab, glint_mask

In [14]:
# --- Cell 3: Fast SLIC + GLCM feature extraction ---
# Dataset used: none - reusable functions for every image below.

def quantize(patch, levels=GLCM_LEVELS):
    return (patch.astype(np.float32) / 256 * levels).astype(np.uint8)

def extract_superpixel_features(img_lab, glint_mask, n_segments=N_SEGMENTS):
    """
    Runs SLIC on the L channel, then computes GLCM texture stats per
    superpixel, keeping true 2D spatial structure. Uses ndimage.find_objects
    for speed and quantized grey levels to shrink the GLCM matrix size.
    """
    L_channel = img_lab[:, :, 0]
    segments = slic(img_lab, n_segments=n_segments, compactness=10, start_label=1)
    objects = ndimage.find_objects(segments)

    features = []
    for seg_id, bbox in enumerate(objects, start=1):
        if bbox is None:
            continue
        y_slice, x_slice = bbox
        local_mask = segments[y_slice, x_slice] == seg_id

        if glint_mask[y_slice, x_slice][local_mask].mean() > 0.5:
            continue  # skip glint-dominated superpixels entirely

        patch = L_channel[y_slice, x_slice]
        if patch.shape[0] < 2 or patch.shape[1] < 2:
            continue

        patch_q = quantize(patch)
        glcm = graycomatrix(patch_q, distances=[1], angles=[0],
                             levels=GLCM_LEVELS, symmetric=True, normed=True)

        features.append({
            "seg_id": int(seg_id),
            "bbox": (x_slice.start, y_slice.start, x_slice.stop, y_slice.stop),
            "contrast": float(graycoprops(glcm, 'contrast')[0, 0]),
            "homogeneity": float(graycoprops(glcm, 'homogeneity')[0, 0]),
            "energy": float(graycoprops(glcm, 'energy')[0, 0]),
        })
    return segments, features

def process_one_image(path):
    """Wrapper for parallel/serial use - returns just the feature list."""
    img, lab, glint_mask = preprocess_image(path)
    _, feats = extract_superpixel_features(lab, glint_mask)
    return feats

In [15]:
# --- Cell 4: Build the normal-water baseline from ALL baseline images (serial version) ---
# Dataset used: normal sea - EVERY image in the baseline split.

all_baseline_paths = list(BASELINE_DIR.glob("*.jpg"))
print(f"Building baseline from ALL {len(all_baseline_paths)} normal sea images...")

baseline_stats = {"contrast": [], "homogeneity": [], "energy": []}

for i, path in enumerate(all_baseline_paths):
    feats = process_one_image(path)
    for f in feats:
        for k in baseline_stats:
            baseline_stats[k].append(f[k])
    if (i + 1) % 200 == 0:
        print(f"  processed {i + 1}/{len(all_baseline_paths)} images")

baseline_ref = {k: (np.mean(v), np.std(v)) for k, v in baseline_stats.items()}
print(f"Baseline built from all {len(all_baseline_paths)} images")
print("Baseline (mean, std):", baseline_ref)

Building baseline from ALL 3283 normal sea images...
  processed 200/3283 images
  processed 400/3283 images
  processed 600/3283 images
  processed 800/3283 images
  processed 1000/3283 images
  processed 1200/3283 images
  processed 1400/3283 images
  processed 1600/3283 images
  processed 1800/3283 images
  processed 2000/3283 images
  processed 2200/3283 images
  processed 2400/3283 images
  processed 2600/3283 images
  processed 2800/3283 images
  processed 3000/3283 images
  processed 3200/3283 images
Baseline built from all 3283 images
Baseline (mean, std): {'contrast': (np.float64(0.3578894036855254), np.float64(0.7279432777124815)), 'homogeneity': (np.float64(0.8765976817912816), np.float64(0.07881566146542675)), 'energy': (np.float64(0.41482075438859517), np.float64(0.16187713241824805))}


In [16]:
# --- Cell 5: Deviation scoring + detector runner ---
# Dataset used: none - reusable scorer for the runs below.

def score_region_deviation(feat, baseline_ref, z_thresh=Z_THRESH):
    z_scores = {
        k: abs(feat[k] - baseline_ref[k][0]) / (baseline_ref[k][1] + 1e-6)
        for k in ["contrast", "homogeneity", "energy"]
    }
    max_z = max(z_scores.values())
    return max_z > z_thresh, max_z

def run_detector(img_dir, baseline_ref, sample_size=None,
                  extensions=("*.jpg", "*.jpeg", "*.png"),
                  min_component_size=MIN_COMPONENT_SIZE,
                  z_thresh=Z_THRESH):
    paths = []
    for ext in extensions:
        paths.extend(Path(img_dir).glob(ext))

    if sample_size:
        paths = random.sample(paths, min(sample_size, len(paths)))

    if len(paths) == 0:
        raise ValueError(f"No images found in {img_dir} - check path and file extensions")

    results = {}
    for path in paths:
        img, lab, glint_mask = preprocess_image(path)
        segments, feats = extract_superpixel_features(lab, glint_mask)
        flagged = [f for f in feats if score_region_deviation(f, baseline_ref, z_thresh)[0]]
        is_detected = len(flagged) >= min_component_size
        results[path.name] = {
            "n_flagged": len(flagged),
            "is_detected": is_detected,
            "flagged_regions": flagged,
        }
    return results

def safe_detection_rate(results, label=""):
    if len(results) == 0:
        raise ValueError(f"No results to compute detection rate for: {label}")
    return np.mean([1 if r["is_detected"] else 0 for r in results.values()])

In [17]:
# --- Cell 6: Run on ALL negative controls (no sample cap) ---
# Dataset used: normal sea - EVERY image in the held-out split.
# This never overlaps with the baseline images from Cell 4.

negative_results = run_detector(NEGATIVE_DIR, baseline_ref, sample_size=None)
false_positive_rate = safe_detection_rate(negative_results, "negative controls")
print(f"False-positive rate on plume-free water (n={len(negative_results)}): {false_positive_rate:.2%}")

False-positive rate on plume-free water (n=457): 25.38%


In [26]:
# --- Cell 7: Run on prawn positives (the only crosscheck for this run) ---
# Dataset used: prawn dataset - 42 deduplicated unique images.

positive_results = run_detector(PRAWN_DIR, baseline_ref)
prawn_detection_rate = safe_detection_rate(positive_results, "prawn positives")
print(f"Detection rate on prawn plumes (n={len(positive_results)}): {prawn_detection_rate:.2%}")

Detection rate on prawn plumes (n=82): 90.24%


In [27]:
# --- Cell 8: Cache raw features for negatives and prawn positives ---
# Dataset used: normal sea (full heldout split) and prawn (unique 42).
# Extracts features once so the sweeps below can re-score cheaply without
# re-running SLIC/GLCM every time.

def run_and_cache_features(img_dir, extensions=("*.jpg", "*.jpeg", "*.png")):
    paths = []
    for ext in extensions:
        paths.extend(Path(img_dir).glob(ext))
    cache = {}
    for path in paths:
        img, lab, glint_mask = preprocess_image(path)
        _, feats = extract_superpixel_features(lab, glint_mask)
        cache[path.name] = feats
    return cache

neg_cache = run_and_cache_features(NEGATIVE_DIR)
pos_cache = run_and_cache_features(PRAWN_DIR)
print(f"Cached features for {len(neg_cache)} negative images and {len(pos_cache)} prawn images")

Cached features for 457 negative images and 82 prawn images


In [28]:
# --- Cell 9: Threshold sweep (z_thresh) ---
# Dataset used: cached negative + prawn features from Cell 8.

def rate_at(cache, z, min_size):
    return np.mean([
        1 if sum(1 for f in feats if score_region_deviation(f, baseline_ref, z)[0]) >= min_size else 0
        for feats in cache.values()
    ])

print(f"{'z_thresh':>10} | {'false_positive_rate':>20} | {'prawn_detection_rate':>20}")
for z in [1.5, 2.0, 2.5, 3.0, 3.5, 4.0]:
    neg_rate = rate_at(neg_cache, z, MIN_COMPONENT_SIZE)
    pos_rate = rate_at(pos_cache, z, MIN_COMPONENT_SIZE)
    print(f"{z:>10} | {neg_rate:>19.2%} | {pos_rate:>19.2%}")

  z_thresh |  false_positive_rate | prawn_detection_rate
       1.5 |              49.67% |             100.00%
       2.0 |              36.11% |              96.34%
       2.5 |              25.38% |              90.24%
       3.0 |              18.38% |              82.93%
       3.5 |              14.22% |              75.61%
       4.0 |              10.28% |              60.98%


In [29]:
# --- Cell 10: Component-size sweep (independent of z_thresh) ---
# Dataset used: cached negative + prawn features from Cell 8.

print(f"{'min_component_size':>20} | {'false_positive_rate':>20} | {'prawn_detection_rate':>20}")
for min_size in [3, 5, 8, 10, 15]:
    neg_rate = rate_at(neg_cache, Z_THRESH, min_size)
    pos_rate = rate_at(pos_cache, Z_THRESH, min_size)
    print(f"{min_size:>20} | {neg_rate:>19.2%} | {pos_rate:>19.2%}")

  min_component_size |  false_positive_rate | prawn_detection_rate
                   3 |              25.38% |              90.24%
                   5 |              21.88% |              85.37%
                   8 |              19.26% |              75.61%
                  10 |              17.51% |              67.07%
                  15 |              15.32% |              56.10%


In [30]:
# --- Cell 11: Inspect false positives directly ---
# Dataset used: normal sea - heldout split (the false-positive images
# from Cell 6's negative_results).

print("Sample false positives (image, n_flagged, bounding boxes of flagged regions):")
shown = 0
for name, r in negative_results.items():
    if r["is_detected"]:
        print(name, r["n_flagged"], [f["bbox"] for f in r["flagged_regions"]])
        shown += 1
    if shown >= 5:
        break

if shown == 0:
    print("No false positives found in negative_results - re-run Cell 6 first.")

Sample false positives (image, n_flagged, bounding boxes of flagged regions):
DSC_0227_FA_1920_224x224_HF_HE_331x331_JPG.rf.d9c5c29f59e5477ceef74c66ed197868.jpg 17 [(310, 0, 396, 52), (301, 0, 474, 161), (402, 0, 498, 46), (422, 50, 512, 121), (299, 93, 512, 336), (444, 167, 512, 218), (451, 202, 512, 269), (398, 256, 512, 320), (419, 315, 512, 367), (418, 348, 512, 397), (448, 387, 512, 461), (251, 388, 482, 460), (345, 450, 467, 512), (284, 452, 405, 496), (452, 457, 512, 512), (249, 467, 357, 512), (172, 481, 266, 512)]
DSC_0227_FA_2560_224x224_HE_331x331_JPG.rf.4522fa6069ff0a5d428ee554c0ab3b9b.jpg 15 [(61, 21, 274, 182), (0, 34, 65, 101), (29, 44, 112, 126), (0, 88, 126, 195), (0, 160, 79, 227), (0, 201, 109, 308), (0, 287, 76, 337), (0, 325, 150, 438), (69, 421, 168, 483), (0, 424, 75, 468), (130, 437, 242, 512), (22, 446, 154, 512), (0, 465, 76, 512), (158, 472, 426, 512), (407, 477, 512, 512)]
DSC_0227_FB_2080_224x224_HM_GN_331x331_JPG.rf.0c87beac6db43393a0f9bf0f66a2a570.jpg 14 

In [31]:
# --- Cell 11b: Visualize a false positive (optional, needs matplotlib) ---

import matplotlib.pyplot as plt
import matplotlib.patches as patches

def visualize_flagged(img_dir, filename, flagged_regions):
    img_path = Path(img_dir) / filename
    img = load_and_resize(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    fig, ax = plt.subplots(1, figsize=(6, 6))
    ax.imshow(img_rgb)
    for f in flagged_regions:
        x0, y0, x1, y1 = f["bbox"]
        rect = patches.Rectangle((x0, y0), x1 - x0, y1 - y0,
                                  linewidth=1.5, edgecolor='red', facecolor='none')
        ax.add_patch(rect)
    ax.set_title(f"Flagged regions: {filename}")
    ax.axis('off')
    plt.show()

for name, r in negative_results.items():
    if r["is_detected"]:
        visualize_flagged(NEGATIVE_DIR, name, r["flagged_regions"])
        break


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\bcura\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\bcura\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\bcura\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\Users\bcura\anaconda3\Lib\site-pack

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ImportError: initialization failed

In [ ]:
# --- Cell 12: Combine into the scored summary (the deliverable) ---
# Dataset used: all of them.
# If the sweeps in Cells 9-10 point to a better Z_THRESH / MIN_COMPONENT_SIZE,
# update Cell 1, then re-run Cells 6, 7 and this cell for the final numbers.

summary = {
    "baseline_images_used": len(all_baseline_paths),
    "z_thresh_used": Z_THRESH,
    "min_component_size_used": MIN_COMPONENT_SIZE,
    "false_positive_rate": false_positive_rate,
    "prawn_detection_rate": prawn_detection_rate,
    "n_negative_images": len(negative_results),
    "n_prawn_images": len(positive_results),
}
print(json.dumps(summary, indent=2))

with open("stage1_scored_baseline.json", "w") as f:
    json.dump(summary, f, indent=2)